In [ ]:
import fsspec
import pandas as pd

root_dir = 'abfss://datalakemedallioncontainer@edai09datalakemedallion.dfs.core.windows.net/'
source = root_dir+"gold/processing/"

# orgainize silver data by location
silver_dfs = {
    "schwartau": pd.DataFrame(),
    "wurzburg": pd.DataFrame()
}
# empty list to hold merged DataFrames
gold_dfs = []

# list files in source
# 'abfs': azure blob file system
fs = fsspec.filesystem("abfs")
file_list_longform = fs.ls(source)
file_list = [file.rsplit("/", maxsplit=1)[1] for file in file_list_longform]

for file in file_list:
    # skip "hidden" files
    if file.startswith("."):
        continue

    print(f"processing {file}")
    df = pd.read_parquet(source+file, filesystem=fs)
    location = file.split("_")[0]

    # temperature and humidity come from hive data and weather station: clarify source
    if df.shape[1] == 2:
        if "temperature" in df.columns:
            df.columns = ["timestamp", "temperature_hive"]
        elif "humidity" in df.columns:
            df.columns = ["timestamp", "humidity_hive"]

    if silver_dfs[location].empty:
        silver_dfs[location] = df.copy()
    else:
        silver_dfs[location] = silver_dfs[location].merge(df, on="timestamp", how="outer")

for loc in silver_dfs:
    # skip locations with no DataFrames
    if silver_dfs[loc].empty:
        continue
    # add location to DataFrames
    silver_dfs[loc]["location"] = loc
    silver_dfs[loc]["location"] = silver_dfs[loc]["location"].astype("category")
    # sort DataFrame by timestamps
    silver_dfs[loc] = silver_dfs[loc].sort_values(by="timestamp")

gold = pd.concat((silver_dfs["schwartau"], silver_dfs["wurzburg"]), ignore_index=True)[['timestamp', 'location', 'flow_out', 'flow_in',
       'temperature_hive', 'humidity_hive', 'weight', 'precipitation', 'pressure_msl',
       'sunshine', 'temperature', 'wind_direction', 'wind_speed',
       'cloud_cover', 'dew_point', 'relative_humidity', 'wind_gust_direction',
       'wind_gust_speed', 'solar']]


gold

StatementMeta(dsai04instruct, 15, 3, Finished, Available, Finished)

processing schwartau_flow__2025-12-16T10h34m51s.611651.parquet
processing schwartau_humidity__2025-12-16T10h34m52s.164823.parquet
processing schwartau_temperature__2025-12-16T10h34m52s.456886.parquet
processing schwartau_weather__2025-12-16T10h35m41s.294336.parquet
processing schwartau_weight__2025-12-16T10h34m52s.716368.parquet


,timestamp,location,flow_out,flow_in,temperature_hive,humidity_hive,weight,precipitation,pressure_msl,sunshine,...,temperature_source_distance,wind_direction_source_distance,wind_speed_source_distance,cloud_cover_source_distance,dew_point_source_distance,relative_humidity_source_distance,visibility_source_distance,wind_gust_direction_source_distance,wind_gust_speed_source_distance,solar_source_distance
0,2017-01-01 12:00:00+00:00,schwartau,NaN,NaN,NaN,98.04,50736.79,0.0,1013.8,0.0,...,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0
1,2017-01-01 13:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,0.0,1012.0,0.0,...,22269.0,22269.0,22269.0,22269.0,22269.0,22269.0,22269.0,22269.0,22269.0,2122.0
2,2017-01-01 13:10:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-01-01 13:15:00+00:00,schwartau,0.0,0.0,12.34,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-01-01 13:16:00+00:00,schwartau,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1258966,2019-05-31 12:13:00+00:00,schwartau,-208.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1258967,2019-05-31 12:14:00+00:00,schwartau,-217.0,146.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1258968,2019-05-31 12:15:00+00:00,schwartau,-222.0,164.0,34.57,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1258969,2019-05-31 13:00:00+00:00,schwartau,NaN,NaN,NaN,NaN,NaN,0.0,1022.0,0.0,...,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0,13018.0


In [ ]:
# write to file
sink = root_dir+"gold/"

write_name = sink+f"hivedata__{pd.Timestamp.now().strftime('%Y-%m-%dT%Hh%Mm%Ss')}.parquet"
with fs.open(write_name, "wb") as f:
    gold.to_parquet(f, index=False)


In [ ]:
gold_cl = gold.set_index('timestamp').groupby('location').resample('d').mean().reset_index()
gold_cl.describe()

In [ ]:
# write to file
sink = root_dir+"gold/"

write_name = sink+f"hivedata_clean__{pd.Timestamp.now().strftime('%Y-%m-%dT%Hh%Mm%Ss')}.parquet"
with fs.open(write_name, "wb") as f:
    gold_cl.to_parquet(f, index=False)


# Release Spark pool

In [ ]:
mssparkutils.session.stop()

# Local

In [37]:
# # local 1
# import pandas as pd

# root_dir = '/Users/karimelbana/wbs/beehaven-azure/'
# source = root_dir+"gold/processing/"

# # orgainize silver data by location
# silver_dfs = {
#     "schwartau": pd.DataFrame(),
#     "wurzburg": pd.DataFrame()
# }
# # empty list to hold merged DataFrames
# gold_dfs = []


In [36]:
# # local 2

# from pathlib import Path

# folder = Path(source)
# file_list = []
# for file in folder.iterdir():
#     if file.is_file():
#         print(file.name)
#         file_list.append(file.name)

# for file in file_list:
#     # skip "hidden" files
#     if file.startswith("."):
#         continue

#     print(f"processing {file}")
#     df = pd.read_parquet(source+file)
#     location = file.split("_")[0]

#     # temperature and humidity come from hive data and weather station: clarify source
#     if df.shape[1] == 2:
#         if "temperature" in df.columns:
#             df.columns = ["timestamp", "temperature_hive"]
#         elif "humidity" in df.columns:
#             df.columns = ["timestamp", "humidity_hive"]

#     if silver_dfs[location].empty:
#         silver_dfs[location] = df.copy()
#     else:
#         silver_dfs[location] = silver_dfs[location].merge(df, on="timestamp", how="outer")

In [35]:
# # local 3

# for loc in silver_dfs:
#     # skip locations with no DataFrames
#     if silver_dfs[loc].empty:
#         continue
#     # add location to DataFrames
#     silver_dfs[loc]["location"] = loc
#     silver_dfs[loc]["location"] = silver_dfs[loc]["location"].astype("category")
#     # sort DataFrame by timestamps
#     silver_dfs[loc] = silver_dfs[loc].sort_values(by="timestamp")

# # combine silver DataFrames
# gold = pd.concat((silver_dfs["schwartau"], silver_dfs["wurzburg"]), ignore_index=True)[['timestamp', 'location', 'flow_out', 'flow_in',
#        'temperature_hive', 'humidity_hive', 'weight', 'precipitation', 'pressure_msl',
#        'sunshine', 'temperature', 'wind_direction', 'wind_speed',
#        'cloud_cover', 'dew_point', 'relative_humidity', 'wind_gust_direction',
#        'wind_gust_speed', 'solar']]

# gold

In [38]:
# # local 4

# # write to file
# sink = root_dir+"gold/"

# write_name = sink+f"hivedata__{pd.Timestamp.now().strftime('%Y-%m-%dT%Hh%Mm%Ss')}.parquet"
# gold.to_parquet(write_name, index=False)


In [39]:
# # local 5

# gold_cl = gold.set_index('timestamp').groupby('location').resample('d').mean().reset_index()
# gold_cl.describe()

In [ ]:
# # local 6

# # write to file
# sink = root_dir+"gold/"

# write_name = sink+f"hivedata_clean__{pd.Timestamp.now().strftime('%Y-%m-%dT%Hh%Mm%Ss')}.parquet"
# gold_cl.to_parquet(write_name, index=False)
